## Método de Validação cruzada (cross-validation)

### **O que é?**

A validação cruzada é empregada quando o conjunto de dados (treino, validação e teste) é muito limitado, visando utilizar o máximo de informação possível do conjunto de dados disponível para treinamento do modelo. Um detalhe importante: caso o conjunto de validação seja muito pequeno, os hiperparêmetros ficarão com certo ruído.

A técnica consiste em dividir o conjunto de dados em $S$ grupos de tamhanho próximo - no caso do problema de classificação, a divisão deverá coneservar a proporção entre as classes também - utilizando $S-1$ grupos para treinamento e $1$ grupo para teste. Além disso, repetimos todo o processo até todos os grupos serem usados como grupo de teste. Assim, o score de performance é uma média dos scores das $S$ repetições. Quando o conjunto de dados é muito escasso, fazemos $S$ como sendo o número de amostras.

### **Exemplo**

In [64]:
# Bibliotecas usadas
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [65]:
# Carregando dados
diabet_dataset = load_diabetes(return_X_y=False, as_frame=False)
diabet_dataset
X = diabet_dataset["data"]
y = diabet_dataset["target"]

In [ ]:
def cross_validation(model, X: np.array, y: np.array, n_splits: int, problem: str="regression") -> float:

    # Armazenando os scores para repetição
    scores = []

    # Definindo o vetor de predição para reconstruir na iteração
    y_pred = np.zeros(len(y))


    # Definindo o objeto para a cross-validation
    if problem.lower() == "classification":
        kf = StratifiedKFold(n_splits=n_splits)
    else:
        kf = KFold(n_splits=n_splits)

    # Iterando para cada grupo S vezes
    for train_inx, test_inx in kf.split(X, y):

        # Sepando em treino e teste
        X_train, y_train = X[train_inx], y[train_inx]
        X_test, y_test = X[test_inx], y[test_inx]

        # Colocando no modelo e calculando o score
        model.fit(X_train, y_train)
        y_pred[test_inx] = model.predict(X_test)  # Definindo a coordenada prevista
        score = model.score(X_test, y_test)

        # Colocando na lista para tirar a média
        scores.append(score)

    return np.mean(scores), y_pred

In [67]:
# Usando o score da Regressão linear
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred1 = model.predict(X_test)
score1 = model.score(X_test, y_test)

# Usando o score da cross-validation
n_splits = 50
model = LinearRegression()
score2, y_pred2 = cross_validation(model, X, y, n_splits, "regression")

# Comparando
print("Score usando apenas Regressão Linear: ", score1)
print("Score usando Cross-Validation: ", score2)
print()
print("Vetor de teste: ", y_test)
print("Predição com Regressão Linear: ", y_pred1)
print("Predição com Cross-Validation: ", y_pred2)

Score usando apenas Regressão Linear:  0.44615330914896867
Score usando Cross-Validation:  0.35681857102506315

Vetor de teste:  [275. 170. 192.  94.  59. 140. 210. 270. 258. 265. 217.  53. 173. 141.
  65.  95. 262. 261.  42. 174. 217. 270. 276. 182.  66.  48. 281. 237.
  68. 151.  63.  55. 123. 129.  91. 121.  70. 179. 233. 275. 341. 180.
  64. 187. 179.  37. 263.  61.  88. 272. 131.  93.  63. 244. 158. 277.
 164.  96. 127.  59. 107. 129. 172.  42. 201.  90. 296. 126.  90. 198.
  97. 128.  71. 220. 292. 170. 111. 101.  44.  72. 232. 200. 209. 288.
  52.  39.  91. 146. 202. 259. 166. 168. 110.  96.  51. 122. 150.  99.
 253. 104. 216. 172.  88. 202. 170.  88. 121.  49.  79.  57.  52.  59.
  97. 248.  67. 243. 202. 249.  84.  90. 156.  85. 220. 200.  49. 158.
 140. 242. 113. 262.  53. 175.  67.]
Predição com Regressão Linear:  [260.13258381 137.30057931 213.32281855 103.86205787 122.20695807
 167.9960248  158.86300621 245.50590936 177.00644791 217.72773028
 231.37650195 102.06060658 220.